# ChatGPT API


***
## 1. 패키지 설치

In [2]:
!pip install -q openai gradio
!pip install openai==0.28

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.5/76.5 kB 2.5 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 1.106.1
    Uninstalling openai-1.106.1:
      Successfully uninstalled openai-1.106.1


***
## 2. API key 입력

In [ ]:
import openai

openai.api_key = "OPEN_AI_KEYS"

## 답변 생성 함수
1. role
    - system : 전체
    - user : 사용자
     - assistant : ChatGPT
2. content

In [10]:
def gen(x):
  gpt_prompt = [{
      "role": "system",
      "content": "당신은 똑똑하고 친절한 인공지능 수학 도우미 챗봇입니다. 입력에 대해 짧고 간결하게 친절하게 대답해주세요."
  }]

  gpt_prompt.append({
      "role": 'user',
      "content": x
  })

  gpt_response = openai.ChatCompletion.create(
      model="gpt-3.5-turbo",
      messages=gpt_prompt
  )

  return gpt_response["choices"][0]["message"]["content"]


## 테스트

In [12]:
gen("안녕하세요, 당신은 누구입니까?")

'안녕하세요! 저는 똑똑하고 친절한 인공지능 수학 도우미입니다. 무엇을 도와드릴까요?'

In [13]:
gen("선형대수학은 무엇입니까?")

'선형대수학은 벡터, 행렬, 선형 변환 등을 다루는 수학의 한 분야로, 선형 연립 방정식을 푸는 것부터 고유값과 고유벡터, 행렬의 대각화 등 다양한 개념을 다룹니다. 주로 공간 변환과 관련된 문제들을 해결하는 데 활용됩니다.'

## Gradio

In [14]:
import gradio as gr

def inference(text):
  return gen(text)

demo = gr.Interface(fn=inference, inputs="text", outputs="text")

demo.launch(debug=True, share=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://49fd2a08894bf33246.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://49fd2a08894bf33246.gradio.live


But, 위의 예시로는 챗봇이 맥락을 기억하지 못함!!!

***
## 맥락 기억

In [20]:
import gradio as gr

def predict(input, history):
  history.append({"role": "user", "content": input})

  gpt_response = openai.ChatCompletion.create(
      model="gpt-3.5-turbo",
      messages=history
  )

  response = gpt_response["choices"][0]["message"]["content"]
  history.append({"role": "assistant", "content": response})
  messages = [(history[i]['content'], history[i+1]["content"]) for i in range(1, len(history), 2)]

  return messages, history

with gr.Blocks() as demo:
  chatbot = gr.Chatbot(label="ChatBot")

  state = gr.State([{
      "role": "system",
      "content": "당신은 똑똑하고 친절한 인공지능 수학 도우미 챗봇입니다. 입력에 대해 짧고 간결하게 친절하게 대답해주세요."
  }])

  with gr.Row():
    txt = gr.Textbox(show_label=False, placeholder="챗봇에게 아무거나 물어보세요")

  txt.submit(predict, [txt, state], [chatbot, state])

demo.launch(debug=True, share=True)

/tmp/ipython-input-3570225569.py:18: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label="ChatBot")


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://37ff7a55d0eb4b0377.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://37ff7a55d0eb4b0377.gradio.live
